<a href="https://colab.research.google.com/github/elvissoares/EQE595-SimMol/blob/main/notebooks/7_Integracao_Termodinamica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title **Instalando OpenMM e suas dependências**
#@markdown Levará alguns minutos, por favor, tome um café e aguarde. ;-)
# instalando dependências
import subprocess
import sys
subprocess.run("wget https://repo.anaconda.com/miniconda/Miniconda3-py312_24.5.0-0-Linux-x86_64.sh", shell=True)
subprocess.run("chmod +x Miniconda3-py312_24.5.0-0-Linux-x86_64.sh", shell=True)
subprocess.run("bash ./Miniconda3-py312_24.5.0-0-Linux-x86_64.sh -b -f -p /usr/local", shell=True)

subprocess.run("conda install --channel defaults conda python=3.12 --yes", shell=True)
subprocess.run("conda update --channel defaults --all --yes", shell=True)
subprocess.run("conda config --add channels omnia --add channels conda-forge", shell=True)
subprocess.run("conda install openmm openmmtools --yes", shell=True)
# usamos o python 3.12 pois a T4 está usando essa versão do python
# !which python
# !python --version
_ = (sys.path.append("/usr/local/lib/python3.12/site-packages"))

Analisando quais `Platform` do `OpenMM` estão functionando.

In [2]:
!python -m openmm.testInstallation


OpenMM Version: 8.4
Git Revision: 47684368dbbe4185d068be77d32a962059cfc37c

There are 4 Platforms available:

1 Reference - Successfully computed forces
2 CPU - Successfully computed forces
3 CUDA - Error computing forces with CUDA platform
1 warning generated.
1 warning generated.
4 OpenCL - Successfully computed forces

CUDA platform error: Error loading CUDA module: CUDA_ERROR_UNSUPPORTED_PTX_VERSION (222)

Median difference in forces between platforms:

Reference vs. CPU: 6.29256e-06
Reference vs. OpenCL: 6.74321e-06
CPU vs. OpenCL: 7.88412e-07

All differences are within tolerance.


Carregando as bibliotecas

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from openmm.app import *
from openmm import *
from openmm.unit import *
from openmmtools import alchemy

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX in the same code.          *
******************************************



# Aula Prática 07 - Integração Termodinâmica de Soluto em Água

Autor: [Prof. Elvis do A. Soares](https://github.com/elvissoares)

Contato: [elvis@peq.coppe.ufrj.br](mailto:elvis@peq.coppe.ufrj.br) - [Programa de Engenharia Química, PEQ/COPPE, UFRJ, Brasil](https://www.peq.coppe.ufrj.br/)

---

Qual molécula iremos simular?

In [ ]:
my_molecule = 'caffeine'  # substitua pelo nome da sua molécula

Condição termodinâmica

In [ ]:
Temperatura = 298.15 * kelvin # Temperatura em Kelvin
Pressao = 1 * bar # Pressão em bar

In [ ]:
pdbpath = f'{my_molecule}.pdb'
!wget https://raw.githubusercontent.com/elvissoares/EQE595-SimMol/refs/heads/main/pdbs/{pdbpath}

In [ ]:
pdb = PDBFile(pdbpath)

Escolhendo os arquivos de campo de força `OPLS-AA` gerados pelo LibParGen

https://zarbi.chem.yale.edu/ligpargen/

In [ ]:
!wget https://raw.githubusercontent.com/elvissoares/EQE595-SimMol/refs/heads/main/pdbs/tip3p.xml

In [ ]:
xmlpath = f'{my_molecule}.xml'
!wget https://raw.githubusercontent.com/elvissoares/EQE595-SimMol/refs/heads/main/pdbs/{xmlpath}

In [ ]:
forcefield = ForceField(f'{my_molecule}.xml', 'tip3p.xml')
forcefield.setUseGeometricCombinationRule=True # OPLS-AA usa combinação geométrica

Cria a topolgia com moléculas de água ao redor (solvente)

In [ ]:
modeller = Modeller(pdb.topology, pdb.positions)

modeller.addSolvent(forcefield, model='tip3p', padding= 1.0*nanometer)

PDBFile.writeFile(modeller.topology, modeller.positions, open(f'initial_{my_molecule}.pdb', 'w'))

In [ ]:
system = forcefield.createSystem(modeller.topology, nonbondedMethod=PME, nonbondedCutoff=8.5*angstroms, constraints=HBonds)

### Configuração do Sistema Alquímico

In [ ]:
solute_atoms = [atom.index for atom in modeller.topology.atoms() if atom.residue.name == "UNK"]
print("Solute atoms:", solute_atoms)

alchemical_region = alchemy.AlchemicalRegion(alchemical_atoms = solute_atoms)
solute_system = alchemy.AbsoluteAlchemicalFactory().create_alchemical_system(system, alchemical_region)

In [ ]:
# Plataforma (GPU se disponível)
platform = Platform.getPlatformByName('OpenCL')

# Integrador
integrator = LangevinMiddleIntegrator(Temperatura, 1/picosecond, 4*femtoseconds)
simulation = Simulation(modeller.topology, solute_system, integrator, platform)

In [ ]:
# below corresponds to fully interacting state
simulation.context.setParameter('lambda_electrostatics', 1.0)
simulation.context.setParameter('lambda_sterics', 1.0)

In [ ]:
n_equil = 10_000

# minimização inicial da energia
simulation.context.setPositions(modeller.positions)
simulation.minimizeEnergy()

simulation.context.setVelocitiesToTemperature(Temperatura)

# Equilibração NVT
simulation.step(n_equil)

# Adcionar barostato para NPT
solute_system.addForce(MonteCarloBarostat(Pressao, Temperatura))
simulation.context.reinitialize(preserveState=True)

# Equilibração NPT
simulation.step(n_equil)

initial_positions = simulation.context.getState(getPositions=True).getPositions()

In [ ]:
PDBFile.writeFile(modeller.topology, modeller.positions, open(f'equilibrated_{my_molecule}.pdb', 'w'))

### Loop de Integração Termodinâmica

Somente interação eletrostática

In [ ]:
n_steps = 100 # passos de MD
n_samples = 2500 # amostras da energia

lambda_ele_grid = np.linspace(1.0, 0.0, 11)
U_mean_ele = np.zeros_like(lambda_ele_grid)
U_std_ele = np.zeros_like(lambda_ele_grid)

for i, l in enumerate(lambda_ele_grid):
    print('lambda:',l)
    #retorna posições iniciais para cada lambda
    simulation.context.setPositions(initial_positions)
    #define novo lambda a ser simulado
    simulation.context.setParameter('lambda_electrostatics', l)
    #equilibração para novo lambda
    simulation.step(n_equil)

    #produção para novo lambda
    U_lambda = []
    for iteration in range(n_samples):
        simulation.step(n_steps)
        e = simulation.context.getState(energy=True).getPotentialEnergy().value_in_unit(kilojoules_per_mole)
        U_lambda.append(e)

    np.savetxt(f'lambda_electrostatics_l={l:.1f}.txt', U_lambda)

    U_mean_ele[i] = np.mean(np.array(U_lambda))
    U_std_ele[i] = np.std(np.array(U_lambda))
    print(f"U_λ^ele: {U_mean_ele[i]:.4f} +- {U_std_ele[i]:.4f}")

In [ ]:
plt.plot(lambda_ele_grid, U_mean_ele, marker='o', label='U_λ^ele')
plt.fill_between(lambda_ele_grid, U_mean_ele - U_std_ele, U_mean_ele + U_std_ele, alpha=0.2)
plt.xlabel('Lambda Electrostatics')
plt.ylabel('U_λ^ele (kJ/mol)')

Somente interação dispersiva

In [ ]:
lambda_steric_grid = np.linspace(1.0, 0.0, 11)
U_mean_steric = np.zeros_like(lambda_steric_grid)
U_std_steric = np.zeros_like(lambda_steric_grid)

for i, l in enumerate(lambda_steric_grid):
    print('lambda:',l)
    #retorna posições iniciais para cada lambda
    simulation.context.setPositions(initial_positions)
    #define novo lambda a ser simulado
    simulation.context.setParameter('lambda_sterics', l)
    #equilibração para novo lambda
    simulation.step(n_equil)

    #produção para novo lambda
    U_lambda = []
    for iteration in range(n_samples):
        simulation.step(n_steps)
        e = simulation.context.getState(energy=True).getPotentialEnergy().value_in_unit(kilojoules_per_mole)
        U_lambda.append(e)

    np.savetxt(f'lambda_sterics_l={l:.1f}.txt', U_lambda)

    U_mean_steric[i] = np.mean(np.array(U_lambda))
    U_std_steric[i] = np.std(np.array(U_lambda))
    print(f"U_λ^ste: {U_mean_steric[i]:.4f} +- {U_std_steric[i]:.4f}")

In [ ]:
plt.plot(lambda_steric_grid, U_mean_steric, marker='o', color='C1', label='U_λ^ste')
plt.fill_between(lambda_steric_grid, U_mean_steric - U_std_steric, U_mean_steric + U_std_steric, color='C1', alpha=0.2)
plt.xlabel('$\lambda$')
plt.ylabel('U_λ (kJ/mol)')

Salvando dados em arquivo `.xls`

In [ ]:
df = pd.DataFrame()
df['lambda_ele'] = lambda_ele_grid
df['U_ele'] = U_mean_ele
df['err U_ele'] = U_std_ele

df['lambda_steric'] = lambda_steric_grid
df['U_steric'] = U_mean_steric
df['err U_steric'] = U_std_steric

df.to_excel(f'integracao_termodinamica_{my_molecule}.xlsx',sheet_name='nsteps_100_nsamples_2500', index=False)

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> Faça uma estimativa do custo computacional (tempo de simulação) no seu _hardware_ atual para 10 simulações semelhantes a essa.
>